In [1]:
# Install once if needed:
# %pip install rapidocr-onnxruntime opencv-python pandas

import re
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from rapidocr_onnxruntime import RapidOCR

In [30]:
# Load clean table directly from CSV (no OCR)
in_csv = Path("../data/muestras_sonora.csv")
out_csv = Path("../data/geochem_ocr_raw.csv")  # keep downstream filename

assert in_csv.exists(), f"Input not found: {in_csv.resolve()}"

df_table = pd.read_csv(in_csv)

# Standardize ID column name for downstream steps
if "ID" in df_table.columns and "Sample_ID" not in df_table.columns:
    df_table = df_table.rename(columns={"ID": "Sample_ID"})

# Coerce chemical columns to numeric
for c in df_table.columns:
    if c != "Sample_ID":
        df_table[c] = pd.to_numeric(df_table[c], errors="coerce")

# Save standardized dataset
df_table.to_csv(out_csv, index=False)
print(f"Saved: {out_csv.resolve()}")
print(f"Rows: {len(df_table)}, Cols: {len(df_table.columns)}")

df_table.head()

Saved: C:\Users\acb\geot\data\geochem_ocr_raw.csv
Rows: 17, Cols: 12


,Sample_ID,Na,K,Ca,Mg,Li,Cl,F,SO4,PO4,HCO3,CO3
0,SF11,14000,1636,2411,222,27,27935,41,529,0,647,0
1,SF22,13550,1768,1460,315,62,25500,2,380,0,475,0
2,PE1,7200,916,1943,69,17,15378,23,22,0,570,0
3,C1,2000,43,264,3,4,2920,9,62,0,37,0
4,LHDA3,367,19,142,28,1,581,2,195,0,105,0


In [31]:
# Quick validation of loaded table
required_base_cols = ["Sample_ID", "Na", "K", "Ca", "Mg", "Li", "Cl"]
missing = [c for c in required_base_cols if c not in df_table.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Columns:", list(df_table.columns))
print("Null counts:")
print(df_table.isna().sum())

df_table.head(10)

Columns: ['Sample_ID', 'Na', 'K', 'Ca', 'Mg', 'Li', 'Cl', 'F', 'SO4', 'PO4', 'HCO3', 'CO3']
Null counts:
Sample_ID    0
Na           0
K            0
Ca           0
Mg           0
Li           0
Cl           0
F            0
SO4          0
PO4          0
HCO3         0
CO3          0
dtype: int64


,Sample_ID,Na,K,Ca,Mg,Li,Cl,F,SO4,PO4,HCO3,CO3
0,SF11,14000,1636,2411,222,27,27935,41,529,0,647,0
1,SF22,13550,1768,1460,315,62,25500,2,380,0,475,0
2,PE1,7200,916,1943,69,17,15378,23,22,0,570,0
3,C1,2000,43,264,3,4,2920,9,62,0,37,0
4,LHDA3,367,19,142,28,1,581,2,195,0,105,0
5,SF02C3,2348,195,366,24,5,3449,9,588,0,169,0
6,P013,107,39,70,16,1,145,2,206,0,121,0
7,P033,118,12,71,15,1,200,1,112,0,137,0
8,P043,110,12,82,15,1,160,1,124,0,169,0
9,P053,173,9,44,8,1,214,4,140,0,126,0


In [32]:
# Keep a clean working copy for all downstream calculations
# (already standardized in previous cell)
df_table = df_table.drop_duplicates().reset_index(drop=True)

# Re-save to ensure downstream cells use cleaned table
df_table.to_csv(out_csv, index=False)
print(f"Saved clean table: {out_csv.resolve()}")
print(f"Rows exported: {len(df_table)}")

df_table

Saved clean table: C:\Users\acb\geot\data\geochem_ocr_raw.csv
Rows exported: 17


,Sample_ID,Na,K,Ca,Mg,Li,Cl,F,SO4,PO4,HCO3,CO3
0,SF11,14000,1636,2411,222,27,27935,41,529,0,647,0
1,SF22,13550,1768,1460,315,62,25500,2,380,0,475,0
2,PE1,7200,916,1943,69,17,15378,23,22,0,570,0
3,C1,2000,43,264,3,4,2920,9,62,0,37,0
4,LHDA3,367,19,142,28,1,581,2,195,0,105,0
5,SF02C3,2348,195,366,24,5,3449,9,588,0,169,0
6,P013,107,39,70,16,1,145,2,206,0,121,0
7,P033,118,12,71,15,1,200,1,112,0,137,0
8,P043,110,12,82,15,1,160,1,124,0,169,0
9,P053,173,9,44,8,1,214,4,140,0,126,0


In [33]:
# Retrieve molecular weights (g/mol) and ion charge for species in geochem_ocr_raw.csv
mw_input_csv = Path("../data/geochem_ocr_raw.csv")
mw_output_csv = Path("../data/geochem_molecular_weights.csv")

assert mw_input_csv.exists(), f"Missing input file: {mw_input_csv.resolve()}"

df_geochem = pd.read_csv(mw_input_csv)
species_cols = [c for c in df_geochem.columns if c != "Sample_ID"]

# Standard atomic weights (g/mol)
atomic_weights = {
    "H": 1.008,
    "Li": 6.94,
    "B": 10.81,
    "C": 12.011,
    "O": 15.999,
    "F": 18.998403163,
    "Na": 22.98976928,
    "Mg": 24.305,
    "Si": 28.085,
    "S": 32.06,
    "Cl": 35.45,
    "K": 39.0983,
    "Ca": 40.078,
    "P": 30.973761998,
}

# Ionic charge (valence) used in charge-balance calculations
ion_charges = {
    "Na": 1,
    "K": 1,
    "Ca": 2,
    "Mg": 2,
    "Li": 1,
    "Cl": -1,
    "SO4": -2,
    "HCO3": -1,
    "CO3": -2,
    "PO4": -3,
    "B": 0,
    "F": -1,
    "SiO2": 0,
}

formula_token_re = re.compile(r"([A-Z][a-z]?)(\d*)")

def molar_mass(formula: str) -> float:
    formula = formula.replace("^", "")
    formula = re.sub(r"[+-]\d*$", "", formula)
    formula = formula.replace("+", "").replace("-", "")

    total = 0.0
    parsed = formula_token_re.findall(formula)
    if not parsed:
        raise ValueError(f"Could not parse formula: {formula}")

    for elem, count_str in parsed:
        if elem not in atomic_weights:
            raise KeyError(f"Atomic weight not found for element: {elem}")
        count = int(count_str) if count_str else 1
        total += atomic_weights[elem] * count
    return total

mw_rows = []
for sp in species_cols:
    mass = molar_mass(sp)
    mw_rows.append(
        {
            "species": sp,
            "molecular_weight_g_mol": round(mass, 6),
            "ion_charge": ion_charges.get(sp),
        }
    )

df_mw = pd.DataFrame(mw_rows)
df_mw.to_csv(mw_output_csv, index=False)

print(f"Saved: {mw_output_csv.resolve()}")
df_mw

Saved: C:\Users\acb\geot\data\geochem_molecular_weights.csv


,species,molecular_weight_g_mol,ion_charge
0,Na,22.989769,1
1,K,39.098300,1
2,Ca,40.078000,2
3,Mg,24.305000,2
4,Li,6.940000,1
5,Cl,35.450000,-1
6,F,18.998403,-1
7,SO4,96.056000,-2
8,PO4,94.969762,-3
9,HCO3,61.016000,-1


In [34]:
# Convert mg/dm3 to mmol/dm3 with species in rows and samples in columns
in_csv = Path("../data/geochem_ocr_raw.csv")
assert in_csv.exists(), f"Missing input file: {in_csv.resolve()}"

df_mg = pd.read_csv(in_csv)

# Build a mapping: species -> molecular weight (g/mol)
mw_map = dict(zip(df_mw["species"], df_mw["molecular_weight_g_mol"]))

species = [c for c in df_mg.columns if c != "Sample_ID"]

# Keep only species for which molecular weight is available
species = [s for s in species if s in mw_map]

# Convert to numeric matrix with samples as index
mg_matrix = df_mg.set_index("Sample_ID")[species].apply(pd.to_numeric, errors="coerce")

# mg/dm3 (mg/L) -> mmol/dm3 (mmol/L): mmol/L = mg/L / (g/mol)
mmol_matrix = mg_matrix.copy()
for sp in species:
    mmol_matrix[sp] = mg_matrix[sp] / mw_map[sp]

# Reorient so rows are species and columns are samples
df_mmol = mmol_matrix.T
df_mmol.index.name = "species"

# Optional export
mmol_out_csv = Path("../data/geochem_mmol_dm3.csv")
df_mmol.to_csv(mmol_out_csv)
print(f"Saved: {mmol_out_csv.resolve()}")

df_mmol

Saved: C:\Users\acb\geot\data\geochem_mmol_dm3.csv


Sample_ID,SF11,SF22,PE1,C1,LHDA3,SF02C3,P013,P033,P043,P053,P063,P333,SF33,SF43,SNFDO13,SNFDO23,SFSW3
species,,,,,,,,,,,,,,,,,
Na,608.966536,589.392612,313.182790,86.995219,15.963623,102.132388,4.654244,5.132718,4.784737,7.525086,10.787407,37.190456,555.203491,590.697540,324.579164,324.057193,515.446675
K,41.843251,45.219357,23.428129,1.099792,0.485955,4.987429,0.997486,0.306919,0.306919,0.230189,0.153459,0.588261,31.177826,34.579509,19.182420,18.363970,10.307354
Ca,60.157692,36.428964,48.480463,6.587155,3.543091,9.132192,1.746594,1.771545,2.046010,1.097859,0.648735,2.021059,50.301911,56.689456,42.841459,41.718649,14.322072
Mg,9.133923,12.960296,2.838922,0.123431,1.152026,0.987451,0.658301,0.617157,0.617157,0.329150,0.041144,0.164575,20.571899,19.419872,3.579510,4.566962,56.737297
Li,3.890490,8.933718,2.449568,0.576369,0.144092,0.720461,0.144092,0.144092,0.144092,0.144092,0.144092,0.288184,3.025937,3.170029,2.305476,2.305476,0.288184
Cl,788.011283,719.322990,433.794076,82.369535,16.389281,97.291961,4.090268,5.641749,4.513399,6.036671,8.519041,35.543018,704.372355,726.600846,364.062059,377.038082,542.369535
F,2.158076,0.105272,1.210628,0.473724,0.105272,0.473724,0.105272,0.052636,0.052636,0.210544,0.578996,0.421088,0.210544,0.368452,0.157908,0.052636,0.368452
SO4,5.507204,3.956026,0.229033,0.645457,2.030066,6.121429,2.144582,1.165987,1.290914,1.457483,0.968185,1.936370,7.391522,7.016740,2.779629,1.155576,24.412843
PO4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [35]:
# Convert mmol/dm3 to meq/dm3 using ionic charge
# meq/dm3 = mmol/dm3 * ionic_charge
charge_map = dict(zip(df_mw["species"], df_mw["ion_charge"]))

# Align charges to species index in df_mmol
charge_series = pd.Series(charge_map).reindex(df_mmol.index)

# Signed meq (cations positive, anions negative)
df_meq = df_mmol.mul(charge_series, axis=0)

meq_out_csv = Path("../data/geochem_meq_dm3.csv")
df_meq.to_csv(meq_out_csv)
print(f"Saved: {meq_out_csv.resolve()}")

df_meq

Saved: C:\Users\acb\geot\data\geochem_meq_dm3.csv


Sample_ID,SF11,SF22,PE1,C1,LHDA3,SF02C3,P013,P033,P043,P053,P063,P333,SF33,SF43,SNFDO13,SNFDO23,SFSW3
species,,,,,,,,,,,,,,,,,
Na,608.966536,589.392612,313.182790,86.995219,15.963623,102.132388,4.654244,5.132718,4.784737,7.525086,10.787407,37.190456,555.203491,590.697540,324.579164,324.057193,515.446675
K,41.843251,45.219357,23.428129,1.099792,0.485955,4.987429,0.997486,0.306919,0.306919,0.230189,0.153459,0.588261,31.177826,34.579509,19.182420,18.363970,10.307354
Ca,120.315385,72.857927,96.960926,13.174310,7.086182,18.264384,3.493188,3.543091,4.092021,2.195718,1.297470,4.042118,100.603823,113.378911,85.682918,83.437297,28.644144
Mg,18.267846,25.920592,5.677844,0.246863,2.304053,1.974902,1.316602,1.234314,1.234314,0.658301,0.082288,0.329150,41.143798,38.839745,7.159021,9.133923,113.474594
Li,3.890490,8.933718,2.449568,0.576369,0.144092,0.720461,0.144092,0.144092,0.144092,0.144092,0.144092,0.288184,3.025937,3.170029,2.305476,2.305476,0.288184
Cl,-788.011283,-719.322990,-433.794076,-82.369535,-16.389281,-97.291961,-4.090268,-5.641749,-4.513399,-6.036671,-8.519041,-35.543018,-704.372355,-726.600846,-364.062059,-377.038082,-542.369535
F,-2.158076,-0.105272,-1.210628,-0.473724,-0.105272,-0.473724,-0.105272,-0.052636,-0.052636,-0.210544,-0.578996,-0.421088,-0.210544,-0.368452,-0.157908,-0.052636,-0.368452
SO4,-11.014408,-7.912051,-0.458066,-1.290914,-4.060132,-12.242858,-4.289165,-2.331973,-2.581827,-2.914966,-1.936370,-3.872741,-14.783043,-14.033480,-5.559257,-2.311152,-48.825685
PO4,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000


In [36]:
# Cation and anion sums (meq/dm3) per sample
charge_map = dict(zip(df_mw["species"], df_mw["ion_charge"]))
charges = pd.Series(charge_map).reindex(df_meq.index)

cation_species = charges[charges > 0].index
anion_species = charges[charges < 0].index

# Sums are computed per sample (column-wise)
df_cation_sum = pd.DataFrame([df_meq.loc[cation_species].sum(axis=0)], index=["cation_sum_meq_dm3"])
df_anion_sum = pd.DataFrame([df_meq.loc[anion_species].abs().sum(axis=0)], index=["anion_sum_meq_dm3"])

# Optional exports
cation_out_csv = Path("../data/geochem_cation_sum_meq_dm3.csv")
anion_out_csv = Path("../data/geochem_anion_sum_meq_dm3.csv")
df_cation_sum.to_csv(cation_out_csv)
df_anion_sum.to_csv(anion_out_csv)

print(f"Saved: {cation_out_csv.resolve()}")
print(f"Saved: {anion_out_csv.resolve()}")

df_cation_sum, df_anion_sum

Saved: C:\Users\acb\geot\data\geochem_cation_sum_meq_dm3.csv
Saved: C:\Users\acb\geot\data\geochem_anion_sum_meq_dm3.csv


(Sample_ID                 SF11        SF22         PE1          C1      LHDA3  \
 cation_sum_meq_dm3  793.283509  742.324206  441.699257  102.092553  25.983904   
 
 Sample_ID               SF02C3       P013       P033       P043       P053  \
 cation_sum_meq_dm3  128.079565  10.605612  10.361134  10.562082  10.753387   
 
 Sample_ID                P063      P333        SF33        SF43     SNFDO13  \
 cation_sum_meq_dm3  12.464716  42.43817  731.154874  780.665734  438.908998   
 
 Sample_ID              SNFDO23       SFSW3  
 cation_sum_meq_dm3  437.297858  668.160951  ,
 Sample_ID                SF11        SF22         PE1         C1      LHDA3  \
 anion_sum_meq_dm3  811.787544  735.125157  444.804582  84.740571  22.275544   
 
 Sample_ID              SF02C3       P013       P033      P043       P053  \
 anion_sum_meq_dm3  112.778308  10.467791  10.271671  9.917628  11.227214   
 
 Sample_ID               P063       P333       SF33      SF43    SNFDO13  \
 anion_sum_meq_dm3  12.41

In [37]:
# Final dataframe: charge-balance error (%) per sample
# CBE(%) = ((sum_cations - sum_anions) / (sum_cations + sum_anions)) * 100

sum_c = df_cation_sum.loc["cation_sum_meq_dm3"]
sum_a = df_anion_sum.loc["anion_sum_meq_dm3"]

df_cbe = pd.DataFrame(
    [((sum_c - sum_a) / (sum_c + sum_a)) * 100],
    index=["cbe_percent"],
)

cbe_out_csv = Path("../data/geochem_charge_balance_error_percent.csv")
df_cbe.to_csv(cbe_out_csv)
print(f"Saved: {cbe_out_csv.resolve()}")

df_cbe

Saved: C:\Users\acb\geot\data\geochem_charge_balance_error_percent.csv


Sample_ID,SF11,SF22,PE1,C1,LHDA3,SF02C3,P013,P033,P043,P053,P063,P333,SF33,SF43,SNFDO13,SNFDO23,SFSW3
cbe_percent,-1.152848,0.487262,-0.350289,9.287423,7.684215,6.352816,0.654005,0.433596,3.146796,-2.15566,0.215554,0.518669,0.174374,1.941181,7.117771,5.691303,5.876613


In [38]:
# Flag samples that meet 5% charge-balance threshold
# pass if abs(CBE%) <= 5

df_cbe_flag = pd.DataFrame(index=df_cbe.columns)
df_cbe_flag.index.name = "Sample_ID"
df_cbe_flag["cbe_percent"] = df_cbe.loc["cbe_percent"].values
df_cbe_flag["abs_cbe_percent"] = df_cbe_flag["cbe_percent"].abs()
df_cbe_flag["passes_5pct_threshold"] = df_cbe_flag["abs_cbe_percent"] <= 5

flag_out_csv = Path("../data/geochem_charge_balance_flag_5pct.csv")
df_cbe_flag.to_csv(flag_out_csv)
print(f"Saved: {flag_out_csv.resolve()}")

df_cbe_flag

Saved: C:\Users\acb\geot\data\geochem_charge_balance_flag_5pct.csv


,cbe_percent,abs_cbe_percent,passes_5pct_threshold
Sample_ID,,,
SF11,-1.152848,1.152848,True
SF22,0.487262,0.487262,True
PE1,-0.350289,0.350289,True
C1,9.287423,9.287423,False
LHDA3,7.684215,7.684215,False
SF02C3,6.352816,6.352816,False
P013,0.654005,0.654005,True
P033,0.433596,0.433596,True
P043,3.146796,3.146796,True
